# Why Context Windows Are Not Memory

It is tempting to treat the LLM **context window** as memory: keep appending chat messages and tool outputs until the model "knows" everything. That approach breaks down quickly — on token limits, cost, attention quality, session boundaries, and governance.

**Context** is what the model sees **during one inference**. **Memory** is a managed store you **read from** and **write to** across turns and sessions, injecting only what matters into context.

This notebook demonstrates the failure modes of context-as-memory and contrasts them with a **ShopCo Support Lab** that uses selective recall. You will see:

1. Token budget overflow when stuffing full chat history
2. **Lost-in-the-middle** — facts buried in long context get ignored
3. Truncation dropping early user preferences
4. Session restarts wiping unstored context
5. Memory-backed selective injection staying within budget while preserving key facts

Everything runs offline in plain Python.

In [ ]:
"""
ShopCo Context vs Memory Lab — why context windows are not memory.
"""

import json
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()


def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~4 chars per token for English prose."""
    return max(1, len(text) // 4)


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"order_id": "ORD-1001", "customer": "alice@shop.com", "status": "shipped", "total_usd": 1299.0},
    "ORD-1002": {"order_id": "ORD-1002", "customer": "bob@shop.com", "status": "processing", "total_usd": 449.0},
}

CONTEXT_BUDGET_TOKENS = 512  # deliberately small to make limits visible

print(f"Lab ready — simulated context budget: {CONTEXT_BUDGET_TOKENS} tokens")

---

## 1. Context Is a Buffer, Memory Is a Store

```
  CONTEXT WINDOW (ephemeral, size-limited)
  ┌──────────────────────────────────────┐
  │ system | recalled | messages | tools │  ← exists only for this API call
  └──────────────────────────────────────┘

  MEMORY STORE (persistent, queryable)
  ┌──────────────────────────────────────┐
  │ semantic | episodic | procedural ... │  ← survives sessions; you choose what to load
  └──────────────────────────────────────┘
```

| Property | Context window | Memory store |
|----------|----------------|--------------|
| Lifetime | One inference (or thread until evicted) | Cross-session |
| Size | Hard token cap | Scales with storage tier |
| Selection | Sequential append | Query by user, relevance, type |
| Cost | Every token billed each call | Read/write cost; inject subset only |
| Governance | Hard to audit what's "in the model" | Structured records with metadata |

---

## 2. Simulating a Context Window

We model context as a token budget. Messages append until the budget is exceeded — then something must give (truncate, summarize, or fail).

In [ ]:
@dataclass
class ContextMessage:
    role: str
    content: str

    @property
    def tokens(self) -> int:
        return estimate_tokens(self.content)


class ContextWindow:
    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.messages: list[ContextMessage] = []

    def append(self, role: str, content: str) -> dict[str, Any]:
        msg = ContextMessage(role, content)
        self.messages.append(msg)
        used = self.token_count()
        return {"appended": True, "tokens_used": used, "over_budget": used > self.max_tokens}

    def token_count(self) -> int:
        return sum(m.tokens for m in self.messages)

    def fits(self) -> bool:
        return self.token_count() <= self.max_tokens

    def truncate_oldest(self) -> int:
        """Drop oldest messages until within budget. Returns count removed."""
        removed = 0
        while self.messages and not self.fits():
            self.messages.pop(0)
            removed += 1
        return removed

    def render(self) -> str:
        return "\n".join(f"[{m.role}] {m.content}" for m in self.messages)


ctx = ContextWindow(CONTEXT_BUDGET_TOKENS)
ctx.append("system", "You are ShopCo support.")
print(f"After system prompt: {ctx.token_count()} tokens, fits={ctx.fits()}")

---

## 3. Anti-Pattern: Stuff the Full Chat History

The naive approach appends every user and assistant message forever. Long support threads exceed the budget and burn tokens on every turn.

In [ ]:
def simulate_long_support_thread(n_turns: int = 40) -> dict[str, Any]:
    ctx = ContextWindow(CONTEXT_BUDGET_TOKENS)
    ctx.append("system", "You are ShopCo support. Help with orders and returns.")
    for i in range(n_turns):
        ctx.append("user", f"Turn {i}: Customer asks about order ORD-{1000+i} shipping delay and refund options.")
        ctx.append("assistant", f"Turn {i}: I checked ORD-{1000+i}. Status is processing. Let me know if you need more.")
    return {
        "turns": n_turns,
        "messages_in_context": len(ctx.messages),
        "tokens_used": ctx.token_count(),
        "budget": ctx.max_tokens,
        "over_budget": not ctx.fits(),
        "overflow_tokens": max(0, ctx.token_count() - ctx.max_tokens),
    }


stuffing = simulate_long_support_thread(40)
print(pretty(stuffing))
print(f"\n⚠ {stuffing['overflow_tokens']} tokens over budget — model cannot see full history without truncation or summarization.")

---

## 4. Truncation Drops Early Facts

When context overflows, teams often **drop oldest messages**. The user's identity and preferences from turn 1 vanish — even though the thread "continues."

In [ ]:
def truncation_demo() -> None:
    ctx = ContextWindow(256)  # tight budget forces truncation
    ctx.append("system", "ShopCo support agent.")
    ctx.append("user", "I'm Alice (alice@shop.com). I prefer email. My order is ORD-1001.")
    ctx.append("assistant", "Thanks Alice! I'll use email for updates. Noted ORD-1001.")
    for i in range(60):
        ctx.append("user", f"Follow-up {i}: Any news on shipping for ORD-1001? Need detailed carrier scan timeline.")
        ctx.append("assistant", f"Follow-up {i}: Still monitoring carrier scan. Will update when warehouse confirms dispatch.")
    removed = ctx.truncate_oldest()
    print(f"Truncation removed {removed} oldest messages")
    print(f"Context now fits: {ctx.fits()} ({ctx.token_count()} tokens)")
    has_alice = "alice" in ctx.render().lower()
    has_preference = "email" in ctx.render().lower()
    has_order = "ORD-1001" in ctx.render()
    print(f"\nAfter truncation — Alice in context: {has_alice}")
    print(f"Email preference in context: {has_preference}")
    print(f"ORD-1001 in context: {has_order}")
    if not (has_alice and has_preference):
        print("\n⚠ Early user facts lost — agent may ask Alice to re-identify herself.")


truncation_demo()

---

## 5. Lost in the Middle

Research and production experience show models attend best to the **beginning** and **end** of context. Critical facts buried in the middle of a long transcript are easy to miss — even when they technically fit in the window.

In [ ]:
NEEDLE = "CRITICAL: Customer has a medical-device exemption — never suggest SMS; email only."


def build_haystack_context() -> ContextWindow:
    ctx = ContextWindow(4096)  # larger window — needle still gets "lost" in attention
    ctx.append("system", "ShopCo support.")
    ctx.append("user", "Hi, I need help with ORD-1001.")
    for i in range(30):
        ctx.append("user", f"Filler message {i}: What about tracking update {i}?")
        ctx.append("assistant", f"Filler reply {i}: Checking carrier API.")
    ctx.append("user", NEEDLE)
    for i in range(30, 60):
        ctx.append("user", f"Filler message {i}: More shipping questions.")
        ctx.append("assistant", f"Filler reply {i}: Still on it.")
    ctx.append("user", "What contact channel should you use for me?")
    return ctx


def offline_attention_heuristic(ctx: ContextWindow, query: str) -> str:
    """Simulates weak middle attention: only scans first 3, last 3, and system messages."""
    scanned = ctx.messages[:3] + ctx.messages[-3:]
    scanned_text = " ".join(m.content for m in scanned).lower()
    if "email only" in scanned_text or "medical-device" in scanned_text:
        return "I'll contact you via email only, per your exemption."
    if "sms" in query.lower():
        return "I'll send you an SMS update."  # wrong!
    return "I'll follow up via SMS."  # wrong — missed needle in middle


haystack = build_haystack_context()
needle_in_full = NEEDLE in haystack.render()
reply = offline_attention_heuristic(haystack, "contact channel")
print(f"Needle present in full context: {needle_in_full}")
print(f"Total tokens: {haystack.token_count()}")
print(f"Simulated model reply: {reply}")
print("\n⚠ Fact was in context but not attended to — context ≠ reliable recall.")

---

## 6. Session Boundaries — Context Dies on Restart

Context lives in the running process. When the user returns tomorrow or the pod restarts, unstored chat history is gone. Memory persists independently.

In [ ]:
@dataclass
class SemanticMemory:
    user_id: str
    facts: list[str] = field(default_factory=list)


class PersistentMemoryStore:
    def __init__(self):
        self._users: dict[str, SemanticMemory] = {}

    def write_fact(self, user_id: str, fact: str) -> None:
        if user_id not in self._users:
            self._users[user_id] = SemanticMemory(user_id)
        self._users[user_id].facts.append(fact)

    def recall(self, user_id: str) -> list[str]:
        return self._users.get(user_id, SemanticMemory(user_id)).facts


def session_simulation() -> None:
    # Session 1 — context only, no memory write
    session1_ctx = ContextWindow(1024)
    session1_ctx.append("user", "I'm Alice. VIP customer. Order ORD-1001.")
    session1_ctx.append("assistant", "Welcome back, Alice!")

    # Session 2 — new process, context is empty
    session2_ctx = ContextWindow(1024)
    session2_ctx.append("user", "Hi again — status on my order?")
    ctx_knows_alice = "alice" in session2_ctx.render().lower()
    print(f"Session 2 (context only) knows Alice: {ctx_knows_alice}")

    # Session 1 with memory write
    mem = PersistentMemoryStore()
    mem.write_fact("alice@shop.com", "VIP customer")
    mem.write_fact("alice@shop.com", "Primary order: ORD-1001")

    # Session 2 with memory read
    facts = mem.recall("alice@shop.com")
    session2_ctx = ContextWindow(1024)
    session2_ctx.append("system", "Recalled: " + "; ".join(facts))
    session2_ctx.append("user", "Hi again — status on my order?")
    oid = re.search(r"ORD-\d+", " ".join(facts))
    print(f"Session 2 (memory-backed) recalled: {facts}")
    print(f"Can resolve order without re-asking: {oid is not None}")


session_simulation()

---

## 7. Cost — You Pay for Every Context Token Every Turn

Stuffing history multiplies cost linearly with thread length. Memory injects a **small recalled slice** each turn.

In [ ]:
COST_PER_1K_INPUT = 0.15  # illustrative $/1K tokens


def cost_comparison(n_turns: int = 20) -> dict[str, Any]:
    # History stuffing: each turn sends entire history
    stuffing_tokens_per_turn = []
    history: list[str] = ["system: ShopCo support"]
    for i in range(n_turns):
        history.append(f"user: question {i} about order")
        history.append(f"assistant: answer {i}")
        stuffing_tokens_per_turn.append(estimate_tokens("\n".join(history)))
    stuffing_total = sum(stuffing_tokens_per_turn)

    # Memory-backed: fixed system + small recall block + current message
    recall_block = "User: Alice, VIP, prefers email. Last order ORD-1001."
    memory_tokens_per_turn = estimate_tokens("system: ShopCo\n" + recall_block + "\nuser: current question")
    memory_total = memory_tokens_per_turn * n_turns

    return {
        "turns": n_turns,
        "stuffing_total_tokens": stuffing_total,
        "stuffing_est_cost_usd": round(stuffing_total / 1000 * COST_PER_1K_INPUT, 4),
        "memory_total_tokens": memory_total,
        "memory_est_cost_usd": round(memory_total / 1000 * COST_PER_1K_INPUT, 4),
        "savings_pct": round(100 * (1 - memory_total / stuffing_total), 1),
    }


costs = cost_comparison(20)
print(pretty(costs))
print(f"\nMemory-backed approach uses ~{costs['savings_pct']}% fewer tokens over 20 turns.")

---

## 8. Memory-Backed Context Builder

The production pattern: **query memory**, build a compact context block, append only the current turn. Stay within budget while preserving salient facts.

In [ ]:
@dataclass
class MemoryRecord:
    content: str
    salience: float = 1.0
    tags: list[str] = field(default_factory=list)


class ShopCoMemory:
    def __init__(self):
        self._records: list[MemoryRecord] = []

    def add(self, content: str, salience: float = 1.0, tags: list[str] | None = None) -> None:
        self._records.append(MemoryRecord(content, salience, tags or []))

    def recall(self, query: str, limit: int = 5) -> list[MemoryRecord]:
        q = query.lower()
        scored = []
        for r in self._records:
            hit = q in r.content.lower() or any(q in t for t in r.tags)
            score = r.salience + (2.0 if hit else 0.0)
            scored.append((score, r))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [r for _, r in scored[:limit]]


class MemoryBackedContextBuilder:
    def __init__(self, memory: ShopCoMemory, max_tokens: int):
        self.memory = memory
        self.max_tokens = max_tokens

    def build(self, user_message: str, thread_messages: list[str] | None = None) -> ContextWindow:
        ctx = ContextWindow(self.max_tokens)
        ctx.append("system", "You are ShopCo support.")
        recalled = self.memory.recall(user_message)
        if recalled:
            block = "Recalled memory:\n" + "\n".join(f"- {r.content}" for r in recalled)
            ctx.append("system", block)
        # Only last 2 thread messages — not full history
        for msg in (thread_messages or [])[-2:]:
            ctx.append("user", msg)
        ctx.append("user", user_message)
        if not ctx.fits():
            ctx.truncate_oldest()
        return ctx


mem = ShopCoMemory()
mem.add("Alice (alice@shop.com) prefers email. VIP tier.", salience=2.0, tags=["alice", "preference"])
mem.add("ORD-1001 shipped 2026-07-01. Prior delay complaint.", salience=1.5, tags=["ORD-1001"])
mem.add("Always cite return policy before refund.", salience=2.0, tags=["policy"])

builder = MemoryBackedContextBuilder(mem, CONTEXT_BUDGET_TOKENS)
built = builder.build(
    "Can I get a refund on ORD-1001?",
    thread_messages=["Hi, I'm Alice.", "My package was late last week."],
)
print(f"Built context: {built.token_count()} tokens (budget {built.max_tokens})")
print(f"Fits budget: {built.fits()}")
print("\n--- Context preview ---")
print(built.render())

---

## 9. Side-by-Side: Three Strategies

Compare full history, truncation-only, and memory-backed on the same 30-turn thread.

In [ ]:
def evaluate_strategy(name: str, ctx: ContextWindow, check_facts: dict[str, bool]) -> dict[str, Any]:
    text = ctx.render().lower()
    return {
        "strategy": name,
        "tokens": ctx.token_count(),
        "fits": ctx.fits(),
        "facts_preserved": {k: (v in text if isinstance(v, str) else k in text) for k, v in check_facts.items()},
    }


KEY_FACTS = {"alice": "alice", "email": "email", "ord-1001": "ord-1001"}

# Strategy A: full stuff then truncate
ctx_a = ContextWindow(CONTEXT_BUDGET_TOKENS)
ctx_a.append("system", "ShopCo support")
ctx_a.append("user", "I'm Alice. Email only. Order ORD-1001.")
for i in range(30):
    ctx_a.append("user", f"msg {i}")
    ctx_a.append("assistant", f"reply {i}")
ctx_a.truncate_oldest()

# Strategy B: memory-backed
ctx_b = builder.build("Refund for ORD-1001?", ["msg 29", "msg 30"])

for result in [
    evaluate_strategy("truncate_full_history", ctx_a, KEY_FACTS),
    evaluate_strategy("memory_backed", ctx_b, KEY_FACTS),
]:
    print(pretty(result))
    print()

---

## 10. What Context Is Good For

Context windows are essential — they are how the model **reasons right now**. Use them for:

- The **current user message** and immediate tool results
- A **small** number of recent turns for conversational coherence
- **Injected recall** from memory (compact, curated)
- System instructions and safety policies

Do **not** use context as the sole persistence layer for user identity, preferences, long ticket history, or cross-session state.

---

## 11. Offline Agent — Context-Only vs Memory-Backed

Two minimal agents answer the same multi-day scenario.

In [ ]:
class ContextOnlyAgent:
    def __init__(self):
        self.ctx = ContextWindow(CONTEXT_BUDGET_TOKENS)
        self.ctx.append("system", "ShopCo support")

    def turn(self, message: str) -> str:
        self.ctx.append("user", message)
        self.ctx.truncate_oldest()
        text = self.ctx.render().lower()
        if "ord-1001" in message.lower() and "ord-1001" in text:
            reply = "ORD-1001 is shipped."
        elif "alice" in text and "email" in text:
            reply = "I'll email you."
        else:
            reply = "Please tell me your name and order ID again."
        self.ctx.append("assistant", reply)
        return reply


class MemoryBackedAgent:
    def __init__(self, memory: ShopCoMemory):
        self.memory = memory
        self.builder = MemoryBackedContextBuilder(memory, CONTEXT_BUDGET_TOKENS)
        self.recent: list[str] = []

    def turn(self, message: str) -> str:
        if "alice" in message.lower():
            self.memory.add("User identified as Alice in this thread.", tags=["alice"])
        if "email" in message.lower():
            self.memory.add("User prefers email contact.", salience=2.0, tags=["email"])
        ctx = self.builder.build(message, self.recent)
        self.recent.append(message)
        text = ctx.render().lower()
        if "ord-1001" in message.lower() or "ord-1001" in text:
            reply = "ORD-1001 is shipped ($1299)."
        else:
            reply = "How can I help?"
        if "alice" in text and "email" in text:
            reply += " I'll follow up via email."
        return reply


def run_day2_scenario(agent_name: str, agent: Any) -> None:
    print(f"=== {agent_name} ===")
    for msg in [
        "I'm Alice, email only please. Order ORD-1001.",
        *[f"Follow-up {i}: any tracking news?" for i in range(28)],
        "What's my order status? Use my preferred contact.",
    ]:
        reply = agent.turn(msg)
    print(f"Final reply: {reply}\n")


run_day2_scenario("Context-only (truncates)", ContextOnlyAgent())
run_day2_scenario("Memory-backed", MemoryBackedAgent(ShopCoMemory()))

---

## 12. Anti-Patterns

| Anti-pattern | Why it fails | Better approach |
|--------------|--------------|------------------|
| Append full chat forever | Token overflow, cost explosion | Recall + last N turns |
| Truncate oldest only | Loses identity and preferences | Persist facts to semantic memory |
| Assume "it's in context" | Lost-in-the-middle | Place critical facts in recalled block |
| No cross-session store | User repeats themselves daily | Write episodic/semantic on session end |
| Same context for all users | Privacy leak | Scope memory and recall by `user_id` |
| Skip write after tool call | Tool results vanish next turn | Write structured tool summaries to memory |

---

## 13. Check Your Understanding

1. What is the core difference between **context** and **memory**?
2. Why does stuffing full chat history fail on long threads?
3. What user information is lost when you **truncate oldest** messages?
4. What is the **lost-in-the-middle** problem?
5. Why does context not survive a **process restart**?
6. How does memory-backed context reduce **token cost**?
7. What belongs in context vs what belongs in a memory store?

---

## 14. Summary

The context window is a **limited, ephemeral reasoning surface** — not a database. Stuffing chat history causes overflow, truncation drops early facts, middle content gets ignored, sessions reset, and costs compound every turn.

**Agentic memory** solves persistence and selection: write durable facts and events, recall a compact slice into context, and keep the current message plus a few recent turns. The ShopCo lab showed memory-backed agents preserving Alice's preferences and ORD-1001 across long threads while staying within a 512-token budget.